In [1]:
# Load env variables and create client
import os
from dotenv import load_dotenv
from groq import Groq
import json

load_dotenv()

# Initialize the Groq client and specify the model to use
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
model = "llama-3.3-70b-versatile"

In [20]:
def chat(messages, system=None, temperature=0.0, stop_sequences=[]):
    if system:
        messages = [{"role": "system", "content": system}] + messages

    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop": stop_sequences if stop_sequences else None,
    }

    completion = client.chat.completions.create(**params)
    return completion.choices[0].message.content

In [21]:
# Function to generate a new dataset
def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [22]:
# Function to run a single test case
def run_test_case(test_case):
    prompt = f"""
Please solve the following task:

{test_case['task']}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [23]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [24]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [25]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [26]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [31]:
# Load the dataset
with open("dataset.json", "r") as f:
    dataset = json.load(f)

# RUN the evaluation and assign it to 'results'
results = run_eval(dataset)

# Save the results
with open("results.json", "w") as f:
    json.dump(results, f, indent=2)


Average score: 7.666666666666667


In [32]:
print(json.dumps(results, indent=2))

[
  {
    "output": "### Extracting Region from AWS EC2 Instance Metadata JSON Response\n\nHere's a Python function that extracts the 'Region' from an AWS EC2 instance metadata JSON response.\n\n#### Code\n```python\nimport json\nimport requests\n\ndef extract_region(metadata_url='http://169.254.169.254/latest/meta-data/placement/availability-zone'):\n    \"\"\"\n    Extracts the 'Region' from an AWS EC2 instance metadata JSON response.\n\n    Args:\n        metadata_url (str): The URL of the AWS EC2 instance metadata. Defaults to 'http://169.254.169.254/latest/meta-data/placement/availability-zone'.\n\n    Returns:\n        str: The extracted 'Region'.\n    \"\"\"\n    try:\n        response = requests.get(metadata_url)\n        response.raise_for_status()  # Raise an exception for HTTP errors\n        availability_zone = response.text\n        region = availability_zone[:-1]  # Remove the last character (the zone letter)\n        return region\n    except requests.exceptions.RequestE